In [1]:
# Fit Platt scaling on validation predictions and evaluate it on the held-out test split.
# This matched sample is not population-representative; report outputs as relative risk scores.
# Save calibration metrics, reliability bins, a figure, and an HTML report.


In [2]:
import sys as _sys
from pathlib import Path as _Path
_UTILS_ROOT = next(path for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
                   if (path / "notebook_utils.py").is_file())
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths
# Load saved Elastic Net predictions without refitting the underlying classifier.
from pathlib import Path
import html
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, log_loss,
                             roc_auc_score)

ROOT = Path.cwd().resolve()
if not (ROOT / "artifacts/19_model_elastic_net_logistic/elastic_net_predictions.csv").exists():
    ROOT = ROOT.parent
PREDICTIONS = ROOT / "artifacts/19_model_elastic_net_logistic/elastic_net_predictions.csv"
REPORTS = project_paths(ROOT).reports
FIGURES = project_paths(ROOT).figures
REQUIRED = {"company_id", "prediction_date", "target", "split", "bankruptcy_probability"}

predictions = pd.read_csv(PREDICTIONS, dtype={"company_id": "string"})
missing = REQUIRED - set(predictions.columns)
if missing:
    raise ValueError(f"Elastic Net predictions are missing columns: {sorted(missing)}")
if not {"validation", "test"}.issubset(set(predictions["split"])):
    raise ValueError("Saved predictions must include validation and test rows")
predictions["prediction_date"] = pd.to_datetime(predictions["prediction_date"], errors="raise")
predictions["target"] = pd.to_numeric(predictions["target"], errors="raise").astype(int)
predictions = predictions.rename(columns={"bankruptcy_probability": "raw_relative_risk_score"})
predictions["raw_relative_risk_score"] = pd.to_numeric(predictions["raw_relative_risk_score"], errors="raise")
predictions = predictions.loc[predictions["split"].isin(["validation", "test"])].copy()
if predictions.empty or not predictions["target"].isin([0, 1]).all():
    raise ValueError("Validation and test predictions must have binary targets")
if not predictions["raw_relative_risk_score"].between(0, 1).all():
    raise ValueError("Model scores must lie between zero and one")
if predictions.duplicated(["company_id", "prediction_date"]).any():
    raise ValueError("Prediction rows must be unique company-events")
validation = predictions.loc[predictions["split"].eq("validation")].copy()
test = predictions.loc[predictions["split"].eq("test")].copy()
if validation["target"].nunique() != 2 or test["target"].nunique() != 2:
    raise ValueError("Both classes are needed in validation and test for calibration diagnostics")
print(f"Loaded {len(validation)} validation and {len(test)} test events.")


Loaded 86 validation and 252 test events.


In [3]:
# Fit the Platt transformation on validation scores only.
def clipped_logit(probabilities):
    values = np.clip(np.asarray(probabilities, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(values / (1 - values)).reshape(-1, 1)

platt = LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000)
platt.fit(clipped_logit(validation["raw_relative_risk_score"]), validation["target"])
for frame in [validation, test]:
    frame["platt_score"] = platt.predict_proba(clipped_logit(frame["raw_relative_risk_score"]))[:, 1]

platt_parameters = {
    "method": "Regularized Platt scaling on logit scores",
    "regularization_C": 1.0,
    "intercept": float(platt.intercept_[0]),
    "slope": float(platt.coef_[0, 0]),
    "fit_rows": len(validation),
    "fit_bankruptcies": int(validation["target"].sum()),
    "fit_prevalence": float(validation["target"].mean()),
}
print("Platt scaling was fitted on validation only.")


Platt scaling was fitted on validation only.


In [4]:
# Summarize discrimination, probability error, and calibration slope/intercept.
def calibration_parameters(labels, scores):
    model = LogisticRegression(C=1e6, solver="lbfgs", max_iter=5000)
    model.fit(clipped_logit(scores), np.asarray(labels, dtype=int))
    return float(model.intercept_[0]), float(model.coef_[0, 0])

def summarize(frame, split, score_name, fit_note):
    labels = frame["target"].to_numpy(dtype=int)
    scores = frame[score_name].to_numpy(dtype=float)
    intercept, slope = calibration_parameters(labels, scores)
    return {
        "split": split,
        "score": score_name,
        "evaluation_note": fit_note,
        "events": len(frame),
        "bankrupt_events": int(labels.sum()),
        "sample_prevalence": float(labels.mean()),
        "brier_score": float(brier_score_loss(labels, scores)),
        "log_loss": float(log_loss(labels, scores, labels=[0, 1])),
        "roc_auc": float(roc_auc_score(labels, scores)),
        "pr_auc_average_precision": float(average_precision_score(labels, scores)),
        "calibration_intercept": intercept,
        "calibration_slope": slope,
    }

calibration_results = pd.DataFrame([
    summarize(validation, "validation", "raw_relative_risk_score", "Raw score; validation diagnostic"),
    summarize(validation, "validation", "platt_score", "Platt fit and evaluated on validation; optimistic diagnostic"),
    summarize(test, "test", "raw_relative_risk_score", "Raw score; held-out test"),
    summarize(test, "test", "platt_score", "Platt fit on validation; held-out test"),
])
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
metrics_path = REPORTS / "probability_calibration_metrics.csv"
calibration_results.to_csv(metrics_path, index=False)
print("Saved calibration metrics.")


Saved calibration metrics.


In [5]:
# Draw test reliability curves and retain the underlying quantile-bin summaries.
def reliability_bins(frame, score_name, n_bins=8):
    scores = frame[score_name].to_numpy(dtype=float)
    edges = np.unique(np.quantile(scores, np.linspace(0, 1, n_bins + 1)))
    labels = np.searchsorted(edges[1:-1], scores) if len(edges) > 1 else np.zeros(len(scores), dtype=int)
    table = frame[["target"]].copy()
    table["bin"] = labels
    table["score"] = scores
    result = table.groupby("bin", sort=True).agg(
        events=("target", "size"),
        mean_score=("score", "mean"),
        observed_bankruptcy_rate=("target", "mean"),
    ).reset_index()
    result.insert(0, "score_type", score_name)
    return result

reliability = pd.concat([
    reliability_bins(test, "raw_relative_risk_score"),
    reliability_bins(test, "platt_score"),
], ignore_index=True)
bins_path = REPORTS / "probability_reliability_bins.csv"
reliability.to_csv(bins_path, index=False)

fig, ax = plt.subplots(figsize=(7.5, 6.5), constrained_layout=True)
for score_name, label, color in [
    ("raw_relative_risk_score", "Raw relative risk score", "#1769aa"),
    ("platt_score", "Platt-scaled validation score", "#d46c00"),
]:
    rows = reliability.loc[reliability["score_type"].eq(score_name)]
    ax.plot(rows["mean_score"], rows["observed_bankruptcy_rate"], marker="o", linewidth=2,
            color=color, label=label)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Reference line")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Mean model score", ylabel="Observed bankruptcy fraction",
       title="Test-set reliability: matched sample")
ax.grid(alpha=0.25)
ax.legend()
figure_path = FIGURES / "probability_reliability_diagram.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.close(fig)
print("Saved the test reliability diagram and bin table.")


Saved the test reliability diagram and bin table.


In [6]:
# Export a short report that distinguishes cohort calibration from population probabilities.
def html_table(frame):
    return frame.to_html(index=False, border=0, float_format=lambda value: f"{value:.4f}")

parameter_table = pd.DataFrame([platt_parameters])
report = f"""<!doctype html><html><head><meta charset="utf-8"><title>Probability calibration</title>
<style>body{{font:16px Arial,sans-serif;max-width:1050px;margin:32px auto;line-height:1.5;color:#222}}
table{{border-collapse:collapse;width:100%;font-size:14px}}th,td{{border:1px solid #ccc;padding:7px;text-align:right}}th{{background:#eee}}
figure img{{max-width:100%}}.notice{{padding:14px;background:#fff4d6;border-left:4px solid #d49a00}}</style></head><body>
<h1>Probability calibration diagnostics</h1>
<p class="notice"><strong>Interpretation:</strong> The validation and test cohorts are constructed using matched bankrupt and non-bankrupt firms, so their observed prevalence is not a population bankruptcy rate. Treat model outputs and Platt-scaled outputs as <strong>relative bankruptcy risk scores</strong>, not true population probabilities. A representative validation sample or external prevalence information is needed for population calibration.</p>
<h2>Calibration method</h2>
<p>Platt scaling was fit using only the validation predictions and labels. The held-out test metrics use that fixed transformation. Validation Platt metrics are in-sample calibration diagnostics and are optimistic. No test data were used to fit the transformation.</p>
<h2>Validation-fitted Platt parameters</h2>{html_table(parameter_table)}
<h2>Calibration and score metrics</h2>{html_table(calibration_results)}
<h2>Test reliability diagram</h2><figure><img src="../figures/probability_reliability_diagram.png" alt="Test reliability diagram"></figure>
<p>Calibration slope and intercept come from a logistic regression of the observed label on the logit of each score; ideal values are slope 1 and intercept 0. Brier score and log loss summarize score error within this matched cohort. They do not correct the cohort's prevalence shift.</p>
</body></html>"""
report_path = REPORTS / "probability_calibration.html"
report_path.write_text(report, encoding="utf-8")
print("Saved the calibration report.")


Saved the calibration report.
